# CAIA cohort - survival pipeline driver

Copy of `PROFILE/run_locally.ipynb` with paths repointed at the CAIA prediction_inputs / local_runs trees and the data source swapped to the CAIA parquet.  `CAIA/build_prediction_inputs.py`, `CAIA/cox_aggregated.py`, `CAIA/landmark_xgboost.py`, and `CAIA/dynamic_deephit.py` are thin wrappers that delegate to the PROFILE scripts after injecting CAIA defaults (`--id-col person_id`, `--age-col AGE_AT_DIAGNOSIS`, CAIA `--inputs-dir` / `--output-dir`).

# Run survival pipeline locally

End-to-end driver for the survival pipeline: raw labs → consolidated longitudinal data → prediction inputs → models. Each section invokes the underlying script via `!{PYTHON} …`, so one kernel runs the whole chain.

**Before running:** make sure the active kernel is the `survlatent_ode` conda env (or activate it inline with `conda run -n survlatent_ode python …`). Edit the `Paths` cell to point at your data.

## Paths

Defaults point at the cluster filesystem under `/data/gusev/...`. Override `PROJECT_ROOT`, `DATA`, `INPUTS_DIR`, `OUTPUT_DIR` for local copies.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/data/gusev/USERS/jpconnor/code/CAIA")
SURVIVAL_DIR = PROJECT_ROOT / "COMPASS" / "survival_analysis" / "CAIA"

# CAIA cohort data: parquet is the canonical input (pre-filtered upstream).
NEPC_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/")
DATA           = NEPC_PROJ_PATH / "caia_compass_longitudinal.parquet"

# Downstream survival outputs go under .../survival_analysis/CAIA/
INPUTS_DIR = NEPC_PROJ_PATH / "survival_analysis" / "CAIA" / "prediction_inputs"
OUTPUT_DIR = NEPC_PROJ_PATH / "survival_analysis" / "CAIA" / "local_runs"

os.chdir(PROJECT_ROOT)
INPUTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use the kernel's own interpreter for !python calls.
PYTHON = sys.executable

# BLAS thread caps (mirror the bash scripts so models don't oversubscribe cores).
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

print("python:        ", PYTHON)
print("cwd:           ", os.getcwd())
print("survival_dir:  ", SURVIVAL_DIR)
print("caia_parquet:  ", DATA)
print("inputs_dir:    ", INPUTS_DIR)
print("output_dir:    ", OUTPUT_DIR)

## 1. Smoke-check the CAIA parquet

The CAIA cohort comes pre-filtered (PARPi-excluded, first-treatment + PSA filters applied upstream) and unit-standardized.  There is no equivalent of PROFILE's `longitudinal_data_processing.py` to run; we go straight from parquet to landmark-aggregated inputs.

In [ ]:
import sys
sys.path.insert(0, str(SURVIVAL_DIR.parent))
from helpers.loaders import load_caia_parquet

patient_df, labs_df = load_caia_parquet(DATA, id_col='person_id')
print(f'patients: {len(patient_df):,}')
print(f'lab rows: {len(labs_df):,}')
print(f'platinum events: {int(patient_df["PLATINUM"].sum())} / {len(patient_df)}')
print(f'deaths:          {int(patient_df["DEATH"].sum())} / {len(patient_df)}')
print(f'unique labs:     {labs_df["LAB_NAME"].nunique()}')

## 3. Build prediction inputs

Wipes the cached `aggregated_landmark{D}.csv` / `longitudinal_landmark{D}.csv`
(they still contain sentinel-contaminated lab features from before the fix) and
rebuilds from the freshly re-consolidated longitudinal data. Required after any
change to `longitudinal_prediction_data.csv` or `build_prediction_inputs.py`.


In [ ]:
# Wipe cached per-landmark inputs so they're rebuilt from the fixed raw data.
import os
for pattern in (
    "aggregated_landmark*.csv",
    "longitudinal_landmark*.csv",
    "longitudinal_landmark*_manifest.json",
    "pre_treatment_lab_long_landmark*.csv",
):
    for p in INPUTS_DIR.glob(pattern):
        p.unlink()
        print(f"  removed {p.name}")
for fname in (
    "canonical_labs_train_val.csv",
    "build_manifest.json",
    "split_assignments.csv",
    "landmark_mrn_availability.csv",
):
    p = INPUTS_DIR / fname
    if p.exists():
        p.unlink()
        print(f"  removed {p.name}")
print("  cache cleared\n")


In [ ]:
!{PYTHON} {SURVIVAL_DIR}/build_prediction_inputs.py \
    --data {DATA} \
    --output-dir {INPUTS_DIR} \
    --landmark-days 0 90 \
    --time-unit-days 7 \
    --test-frac 0.20 \
    --val-frac 0.20 \
    --min-patient-coverage 0.20

## 4. Cohort diagnostics post-fix

Spot-checks the rebuilt aggregated CSV for the testosterone fix. The platinum-
positive `Testo max` median should be in the 100–1500 ng/dL range now (was
9,999,999 pre-fix). PSA medians should be unchanged since PSA was already
sentinel-filtered upstream by `compile_prostate_data.py`.


In [ ]:
import pandas as pd, numpy as np

for lm in (0, 90):
    agg_path = INPUTS_DIR / f"aggregated_landmark{lm}.csv"
    if not agg_path.exists():
        print(f"  landmark +{lm}d: aggregated CSV not found, skipping")
        continue
    agg = pd.read_csv(agg_path)

    def find_col(substr, stat):
        return next(
            (c for c in agg.columns if substr.lower() in c.lower() and c.endswith(f"__{stat}")),
            None,
        )

    n_plat = int(agg["PLATINUM"].sum())
    n_death = int(agg["DEATH"].sum())
    print(f"=== landmark +{lm}d  |  n_total={len(agg):,}  n_PLATINUM={n_plat}  n_DEATH={n_death} ===")
    for lab_substr in ("Testosterone", "PSA", "Prostate specific Ag"):
        for stat in ("mean", "last", "max", "min"):
            col = find_col(lab_substr, stat)
            if col is None:
                continue
            for ev in (0, 1):
                sub = agg.loc[agg["PLATINUM"] == ev, col].dropna()
                if sub.empty:
                    continue
                print(
                    f"  {lab_substr:>22s} {stat:5s}  PLAT={ev}:  median={sub.median():>10.2f}  "
                    f"max={sub.max():>12.2f}  n={len(sub):>5}"
                )
            break  # only report once per lab (use whichever variant matched)
    print()


## 5. Run all platinum tasks at landmarks 0 and 90

Runs cox + xgboost + deephit (PLATINUM and competing configs) at both landmarks,
plus the PGS-adjusted Cox sweep for Testosterone + PSA. Skips any task whose
metrics CSV already exists at the expected output path. Set `FORCE_RERUN = True`
to override and rerun everything (do this after a preprocessing-layer fix).


In [ ]:
import subprocess
import time

N_FOLDS = 5
FORCE_RERUN = True  # rerun every task and overwrite previous results

# (model, landmark, config_dir, metrics_filename)
# config_dir is the subfolder under landmark_{D}/ where outputs are written.
# "baseline" = age(+stage)-only benchmark (CAIA has no stage source -> age-only).
TASKS = [
    ("cox",     0,  "both",      "cox_agg_multivariable_metrics.csv"),
    ("cox",     90, "both",      "cox_agg_multivariable_metrics.csv"),
    ("cox",     0,  "baseline",  "cox_agg_baseline_metrics.csv"),
    ("cox",     90, "baseline",  "cox_agg_baseline_metrics.csv"),
    ("xgboost", 0,  "both",      "landmark_xgboost_metrics.csv"),
    ("xgboost", 90, "both",      "landmark_xgboost_metrics.csv"),
    ("xgboost", 0,  "baseline",  "landmark_xgboost_baseline_metrics.csv"),
    ("xgboost", 90, "baseline",  "landmark_xgboost_baseline_metrics.csv"),
    # ("deephit", 0,  "PLATINUM",  "dynamic_deephit_metrics_PLATINUM.csv"),  # deephit excluded for now
    # ("deephit", 90, "PLATINUM",  "dynamic_deephit_metrics_PLATINUM.csv"),  # deephit excluded for now
    # ("deephit", 0,  "competing", "dynamic_deephit_metrics_competing.csv"),  # deephit excluded for now
    # ("deephit", 90, "competing", "dynamic_deephit_metrics_competing.csv"),  # deephit excluded for now
]


def build_command(model, landmark, config_dir, row_output_dir):
    if model == "cox":
        return [
            PYTHON, str(SURVIVAL_DIR / "cox_aggregated.py"),
            "--inputs-dir", str(INPUTS_DIR),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--analysis", config_dir,           # "both"=univariate+multivariable; "baseline"=age(+stage) only
            "--endpoints", "platinum",          # death excluded for now
            "--n-folds", str(N_FOLDS),
        ]
    if model == "xgboost":
        cmd = [
            PYTHON, str(SURVIVAL_DIR / "landmark_xgboost.py"),
            "--inputs-dir", str(INPUTS_DIR),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "platinum",          # death excluded for now
            "--n-folds", str(N_FOLDS),
        ]
        if config_dir == "baseline":
            cmd.append("--baseline")            # age(+stage)-only baseline
        return cmd
    if model == "deephit":
        return [
            PYTHON, str(SURVIVAL_DIR / "dynamic_deephit.py"),
            "--inputs-dir", str(INPUTS_DIR),
            "--output-dir", str(row_output_dir),
            "--landmark-day", str(landmark),
            "--config", config_dir,
            "--n-folds", str(N_FOLDS),
        ]
    raise ValueError(f"Unknown model: {model}")


summary = []
for model, landmark, config_dir, metrics_filename in TASKS:
    row_output_dir = OUTPUT_DIR / model / f"landmark_{landmark}" / config_dir
    metrics_path = row_output_dir / metrics_filename
    tag = f"{model:8s}  landmark_{landmark:<2}  {config_dir}"

    if metrics_path.exists() and not FORCE_RERUN:
        print(f"[skip] {tag}  ->  {metrics_path.relative_to(OUTPUT_DIR)} exists")
        summary.append((tag, "skipped", 0.0))
        continue

    row_output_dir.mkdir(parents=True, exist_ok=True)
    cmd = build_command(model, landmark, config_dir, row_output_dir)
    print(f"[run ] {tag}")
    print("       " + " ".join(cmd))
    t0 = time.time()
    rc = subprocess.call(cmd)
    elapsed = time.time() - t0
    status = "ok" if rc == 0 else f"FAILED (rc={rc})"
    print(f"[done] {tag}  -> {status}  ({elapsed/60:.1f} min)\n")
    summary.append((tag, status, elapsed))

print("\n=== run summary ===")
for tag, status, elapsed in summary:
    print(f"  {tag}  {status:>20s}  {elapsed/60:6.1f} min")

# Optional: PGS-adjusted Cox sweep for Testosterone + PSA. Writes
# OUTPUT_DIR/cox/landmark_{D}/both/cox_pgs_adjusted.csv per landmark.
# This is the analysis where the sentinel/range fix matters most — the
# testosterone HR direction should flip once the 9999999 values are gone.
# PGS-adjusted Cox is PROFILE-only: it needs (a) the PROFILE-side germline
# file (complete_germline_data_df.csv.gz) and (b) labs named 'Testosterone'
# / 'PSA'.  CAIA has neither, so this block is disabled by default here.
RUN_PGS_ADJUSTED = False
if RUN_PGS_ADJUSTED:
    pgs_out_dir = OUTPUT_DIR  # script appends cox/landmark_{N}/both itself
    pgs_out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        PYTHON, str(SURVIVAL_DIR / "cox_pgs_adjusted.py"),
        "--aggregated-csv-pattern", str(INPUTS_DIR / "aggregated_landmark{landmark}.csv"),
        "--output-dir", str(pgs_out_dir),
        "--landmarks", "0", "90",
        "--endpoints", "platinum",  # death excluded for now
        "--target-labs", "Testosterone", "PSA",
        "--feature-stat", "mean",
    ]
    print("[run ] cox_pgs_adjusted")
    print("       " + " ".join(cmd))
    t0 = time.time()
    rc = subprocess.call(cmd)
    print(f"[done] cox_pgs_adjusted -> rc={rc}  ({(time.time()-t0)/60:.1f} min)")

## 6. Inspect outputs

Pulls the headline metrics row from each task's metrics CSV into one comparison table.


In [ ]:
import pandas as pd

rows = []
for model, landmark, config_dir, metrics_filename in TASKS:
    metrics_path = OUTPUT_DIR / model / f"landmark_{landmark}" / config_dir / metrics_filename
    if not metrics_path.exists():
        rows.append({
            "model": model, "landmark": landmark, "config": config_dir,
            "endpoint": "-", "n_test": None, "n_test_events": None,
            "c_index": None, "mean_auc_t": None, "integrated_brier": None,
            "status": "missing",
        })
        continue
    df = pd.read_csv(metrics_path)

    if model == "cox":
        platinum = df[df["endpoint"] == "platinum"].iloc[0]
        rows.append({
            "model": model, "landmark": landmark, "config": config_dir,
            "endpoint": "platinum",
            "n_test": int(platinum["n_test"]),
            "n_test_events": int(platinum["n_events_test"]),
            "c_index": float(platinum["test_c_index"]),
            "mean_auc_t": float(platinum["test_mean_auc_t"]),
            "integrated_brier": float(platinum["test_integrated_brier"]),
            "status": "ok",
        })
    elif model == "xgboost":
        platinum = df[df["endpoint"] == "platinum"].iloc[0]
        rows.append({
            "model": model, "landmark": landmark, "config": config_dir,
            "endpoint": "platinum",
            "n_test": int(platinum["n_test"]),
            "n_test_events": int(platinum["n_test_events"]),
            "c_index": float(platinum["c_index"]),
            "mean_auc_t": float(platinum["mean_auc_t"]),
            "integrated_brier": float(platinum["integrated_brier"]),
            "status": "ok",
        })
    elif model == "deephit":
        for _, r in df.iterrows():
            rows.append({
                "model": model, "landmark": landmark, "config": config_dir,
                "endpoint": str(r["event"]),
                "n_test": int(r["n_test"]),
                "n_test_events": int(r["n_test_events"]),
                "c_index": float(r["c_index"]),
                "mean_auc_t": float(r["mean_auc_t"]),
                "integrated_brier": float(r["integrated_brier"]),
                "status": "ok",
            })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(["endpoint", "landmark", "model", "config"]).reset_index(drop=True)
summary_df